## Simulation of Call Option Prices via SDE

### Inputs
- Final time $T$
- Number of steps $N$
- Initial value $X_0 = x_0$
- Drift function $\mu(x, t)$
- Diffusion function $\sigma(x, t)$

### Euler-Maruyama Discretization
1. Set $\Delta t = T/N$, define grid $t_n = n \Delta t$
2. Initialize $X_0 = x_0$
3. For $n = 0$ to $N - 1$:
    - Sample $\Delta W_n \sim \mathcal{N}(0, \Delta t)$
    - Compute:

      $$
      X_{n+1} = X_n + \mu(X_n, t_n) \Delta t + \sigma(X_n, t_n) \Delta W_n
      $$

## Geometric Brownian Motion (GBM)

GBM SDE:
$$
dX_t = \mu X_t \, dt + \sigma X_t \, dW_t
$$

with closed-form solution:
$$
X_t = X_0 \exp \left( \left( \mu - \frac{\sigma^2}{2} \right)t + \sigma W_t \right)
$$

Euler–Maruyama approximation for GBM:
$$
X_{n+1} = X_n + \mu X_n \Delta t + \sigma X_n \Delta W_n
$$

## Evaluating using Monte Carlo Methods for risk-neutral frameworks
1. Simulate $M$ sample paths $\{ S_T^{(i)} \}_{i=1}^M$ using Euler–Maruyama.

2. Compute option payoffs $f(S_T^{(i)})$ for each path based on the option type.

3. Take the average of payoffs:  
   $$
   \hat{P} = \frac{1}{M} \sum_{i=1}^{M} f(S_T^{(i)}).
   $$

4. Discount to present value:  
   $$
   V_0 = e^{-rT} \cdot \hat{P}.
   $$


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [14]:
def euler_maruyama_paths(S0, T, n_steps, drift, diffusion, n_paths):
    # Simulating the euler-maruyama processes
    dt = T / n_steps

    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0

    for i in range(1, n_steps + 1):
        dW = np.random.normal(0, np.sqrt(dt), n_paths)
        S[:, i] = S[:, i - 1] + drift * S[:, i - 1] * dt + diffusion * S[:, i - 1] * dW

    # Plotting the paths
    plt.figure(figsize=(10, 6))
    colors = ['red', 'green', 'navy']
    for i in range(n_paths):
        plt.plot(np.linspace(0, T, n_steps + 1), S[i], lw=0.8, alpha=0.5, color=colors[i%3])
    plt.title(f'Simulated {n_paths} Paths (drift = {drift:.3f}, diffusion = {diffusion:.3f})')
    plt.xlabel('Time')
    plt.ylabel('Stock Price')
    plt.grid()
    plt.show()
    plt.close()

    return S # (n_paths, n_steps + 1)

def evaluate_option_euler(option_type, S, K, T, rf):
    ST = S[:,-1]
    ST_arith = np.mean(S, axis=1)
    ST_geom = np.exp(np.mean(np.log(S), axis = 1))
    ST_min = np.min(S, axis = 1)

    # Calculating payoff of each option type
    if option_type == "euro_call":
        P = np.maximum(ST - K, 0).mean()

    elif option_type == "arith_asian_call":
        P = np.maximum(ST_arith - K, 0).mean()

    elif option_type == "geom_asian_call":
        P = np.maximum(ST_geom - K, 0).mean()

    elif option_type == "float_lback_call":
        P = np.maximum(ST - ST_min, 0).mean()

    else:
        raise ValueError("option_type must be one of: 'euro_call', 'arith_asian_call', 'geom_asian_call', 'float_lback_call'")

    return np.exp(-rf * T) * P, P

def run_sim(option_type, S0, drift, diffusion, T, n_steps, n_paths, K, rf):
    S = euler_maruyama_paths(S0, T, n_steps, drift, diffusion, n_paths)
    price, payoff = evaluate_option_euler(option_type, S, K, T, rf)
    print(f"Option Price: {price:.2f} \nPayoff: {payoff:.2f}")

In [ ]:
def interactive_option_pricing():
    # display("Please wait a few seconds after changing values for the code to run")
    message = widgets.HTML("<b>Please wait after changing values for the code to run.</b>")
    option_type = widgets.Dropdown(
        options=[('European Call', 'euro_call'),
                 ('Arithmetic Asian Call', 'arith_asian_call'),
                 ('Geometric Asian Call', 'geom_asian_call'),
                 ('Floating Lookback Call', 'float_lback_call')],
        value='euro_call',
        description='Option Type:'
    )
    S0 = widgets.FloatSlider(value=100, min=1, max=500, step=1, description='S0:')
    drift = widgets.FloatSlider(value=0.2, min=-1.0, max=1.0, step=0.05, description='Drift:')
    diffusion = widgets.FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01, description='Diffusion:')
    T = widgets.FloatSlider(value=1.0, min=0.01, max=5.0, step=0.01, description='T:')
    n_steps = widgets.IntSlider(value=1000, min=100, max=10000, step=100, description='Steps:')
    n_paths = widgets.IntSlider(value=10000, min=1, max=100000, step=100, description='Paths:')
    K = widgets.FloatSlider(value=80, min=1, max=500, step=1, description='Strike K:')
    rf = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='r (rf):')

    ui = widgets.VBox([message, option_type, S0, drift, diffusion, T, n_steps, n_paths, K, rf])

    out = widgets.interactive_output(
        run_sim,
        {'option_type': option_type, 'S0': S0, 'drift': drift, 'diffusion': diffusion,
         'T': T, 'n_steps': n_steps, 'n_paths': n_paths, 'K': K, 'rf': rf}
    )
    display(ui, out)

interactive_option_pricing()

Output()